In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

raw_data = {
    'Ip_A': [0.03, 0.03, 0.03, 0.03, 0.03, 0.03, 0.03, 0.03,
             0.035, 0.035, 0.035, 0.035, 0.035, 0.035, 0.035, 0.035,
             0.04, 0.04, 0.04, 0.04, 0.04, 0.04, 0.04, 0.04,
             0.045, 0.045, 0.045, 0.045, 0.045, 0.045, 0.045, 0.045,
             0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05],

    'N_channels': [3, 4, 5, 6, 7, 8, 9, 10] * 5,

    'd_opt_A': [0.01034, 0.0054, 0.00399, 0.00331, 0.00278, 0.00236, 0.00206, 0.00182,
                0.01361, 0.00698, 0.00496, 0.00401, 0.00334, 0.00283, 0.00245, 0.00216,
                0.01688, -0.00858, 0.00597, 0.00472, 0.00389, 0.00328, 0.00282, 0.00248,
                0.02007, 0.01015, 0.00698, 0.00543, 0.00443, 0.00371, 0.00319, 0.00281,
                0.02315, 0.01168, 0.00796, 0.00612, 0.00495, 0.00414, 0.00356, 0.00312],

    'BER': [7.82E-14, 6.03E-5, 0.00248, 0.01007, 0.02478, 0.04615, 0.07118, 0.09721,
            1.25E-22, 3.18E-7, 2.15E-4, 0.00225, 0.00883, 0.02184, 0.04029, 0.06194,
            9.10E-34, 4.52E-10, 1.06E-5, 3.91E-4, 0.0028, 0.00967, 0.0218, 0.03818,
            6.76E-47, 2.09E-13, 3.26E-7, 5.46E-5, 7.97E-4, 0.004, 0.01124, 0.02269,
            1.08E-61, 3.76E-17, 6.70E-9, 6.32E-6, 2.04E-4, 0.00154, 0.0055, 0.01295]
}
data = pd.DataFrame(raw_data)

X = data[['Ip_A', 'N_channels']]
y = data[['BER', 'd_opt_A']]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=3, include_bias=False)),
    ('ridge', Ridge(alpha=1.0))
])

pipeline.fit(X_train, y_train)

model = pipeline.named_steps['ridge']
poly = pipeline.named_steps['poly']

print("\nModel Coefficients:\n")

feature_names = poly.get_feature_names_out(['Ip_A', 'N_channels'])

for i, target in enumerate(['BER', 'd_opt_A']):
    print(f"\nOutput Variable: {target}")
    print("Intercept:", model.intercept_[i])

    for j, fname in enumerate(feature_names):
        print(f"Coefficient for {fname}: {model.coef_[i][j]}")

y_pred = pipeline.predict(X_test)

for i, target in enumerate(['BER', 'd_opt_A']):
    mse = mean_squared_error(y_test.iloc[:, i], y_pred[:, i])
    r2 = r2_score(y_test.iloc[:, i], y_pred[:, i])

    print(f"\nPerformance for {target}:")
    print("MSE:", mse)
    print("R2 Score:", r2)


Model Coefficients:


Output Variable: BER
Intercept: 0.001702004774703069
Coefficient for Ip_A: -0.003670273717434786
Coefficient for N_channels: 0.0074519513927569425
Coefficient for Ip_A^2: 0.004010667109966777
Coefficient for Ip_A N_channels: -0.01039648080459906
Coefficient for N_channels^2: 0.00817909537485754
Coefficient for Ip_A^3: -0.0015519652670213807
Coefficient for Ip_A^2 N_channels: 0.00393694606601163
Coefficient for Ip_A N_channels^2: -0.004240763557401132
Coefficient for N_channels^3: 0.0021460233652993302

Output Variable: d_opt_A
Intercept: 0.0022506155226160295
Coefficient for Ip_A: 0.00023776465779796796
Coefficient for N_channels: 0.001092036071019431
Coefficient for Ip_A^2: 0.0010176945504500254
Coefficient for Ip_A N_channels: -0.0010575680320483209
Coefficient for N_channels^2: 0.0022818056244062225
Coefficient for Ip_A^3: 0.00021890632834594906
Coefficient for Ip_A^2 N_channels: -0.0009348370655240833
Coefficient for Ip_A N_channels^2: 0.0008814548437920668
C